In [8]:
import tensorflow as tf
from tensorflow.keras import layers, Model


def conv_block(x, filters, kernel=3):
    x = layers.Conv2D(filters, kernel, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def depthwise_block(x, filters):
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def residual_block(x, filters):
    shortcut = layers.Conv2D(filters, 1, padding='same')(x)

    x = depthwise_block(x, filters)
    x = depthwise_block(x, filters)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x

def se_block(x, reduction=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // reduction, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

# ------------------------------------------------------------
# IMAGE INPUT BRANCH (STRONG CNN)
# ------------------------------------------------------------
input_img_size = 384    
image_input = layers.Input(shape=(input_img_size, input_img_size, 3), name="image_input")

x = conv_block(image_input, 32)
x = residual_block(x, 32)
x = layers.MaxPooling2D(2)(x)

x = conv_block(x, 64)
x = residual_block(x, 64)
x = layers.MaxPooling2D(2)(x)

x = conv_block(x, 128)
x = residual_block(x, 128)
x = layers.MaxPooling2D(2)(x)

x = conv_block(x, 256)
x = residual_block(x, 256)
x = layers.MaxPooling2D(2)(x)

x = conv_block(x, 512)
x = residual_block(x, 512)

# Squeeze-and-Excitation for better feature calibration
x = se_block(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)

image_features = layers.Dense(512, activation='relu')(x)   # stronger head
image_features = layers.Dropout(0.25)(image_features)

# ------------------------------------------------------------
# METADATA BRANCH
# ------------------------------------------------------------

meta_input = layers.Input(shape=(7,), name="meta_input")

m = layers.Dense(128, activation='relu')(meta_input)
m = layers.BatchNormalization()(m)
m = layers.Dense(64, activation='relu')(m)
m = layers.Dense(32, activation='relu')(m)

# ------------------------------------------------------------
# TARGET-TYPE BRANCH
# ------------------------------------------------------------

target_input = layers.Input(shape=(7,), name="target_type")

t = layers.Dense(32, activation='relu')(target_input)
t = layers.Dense(16, activation='relu')(t)

# ------------------------------------------------------------
# FUSION
# ------------------------------------------------------------

combined = layers.Concatenate()([image_features, m, t])

h = layers.Dense(512, activation='relu')(combined)
h = layers.Dropout(0.3)(h)

h = layers.Dense(256, activation='relu')(h)
h = layers.BatchNormalization()(h)

h = layers.Dense(128, activation='relu')(h)
h = layers.Dropout(0.2)(h)

h = layers.Dense(64, activation='relu')(h)

output = layers.Dense(1)(h)

model = Model(inputs=[image_input, meta_input, target_input], outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 384, 384,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_50 (Conv2D)  │ (None, 384, 384,  │        864 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 384, 384,  │        128 │ conv2d_50[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_100 (ReLU)    │ (None, 384, 384,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_50 │ (None, 384, 384,  │      1,312 │ re_lu_100[0][0]   │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 384, 384,  │        128 │ separable_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_101 (ReLU)    │ (None, 384, 384,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_51 │ (None, 384, 384,  │      1,312 │ re_lu_101[0][0]   │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 384, 384,  │        128 │ separable_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_102 (ReLU)    │ (None, 384, 384,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_51 (Conv2D)  │ (None, 384, 384,  │      1,056 │ re_lu_100[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_25 (Add)        │ (None, 384, 384,  │          0 │ re_lu_102[0][0],  │
│                     │ 32)               │            │ conv2d_51[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_103 (ReLU)    │ (None, 384, 384,  │          0 │ add_25[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_20    │ (None, 192, 192,  │          0 │ re_lu_103[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_52 (Conv2D)  │ (None, 192, 192,  │     18,432 │ max_pooling2d_20… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 192, 192,  │        256 │ conv2d_52[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_104 (ReLU)    │ (None, 192, 192,  │          0 │ batch_normalizat

 Total params: 3,415,313 (13.03 MB)

 Trainable params: 3,408,593 (13.00 MB)

 Non-trainable params: 6,720 (26.25 KB)

In [10]:
from tensorflow.keras.utils import plot_model
plot_model(model, show_shapes=True, expand_nested=True)